# Week 21: High-Performance ML Inference, Compilers & Serving
## Notebook 04: Continuous (Iteration-Level) Batching from Scratch

### 1. Conceptual Intuition & Real-World Use Case
In production LLM APIs, users submit requests with vastly different lengths:
- User A asks: *"What is 2+2?"* (generates 1 token).
- User B asks: *"Write a 1,000-word essay."* (generates 1,000 tokens).

**Static Batching (Traditional ML):**
- Groups User A and User B into a single batch.
- User A finishes in 1 iteration, but the GPU **remains blocked for 999 iterations**, processing empty padding tokens for User A until User B finishes!

In 2022, Yu et al. (Orca / vLLM) introduced **Continuous (Iteration-Level) Batching**:
- The batch schedule operates at the **individual token iteration level**, not the request level!
- As soon as User A emits an EOS (End of Sequence) token, their slot is immediately evacuated, and a brand-new user from the waiting queue is inserted on the very next token step!

---

### 2. Architecture & Queue Dynamics
$$\text{Iteration } t: [\text{Req 1 (step 50)}, \text{Req 2 (step 1 - EOS)}] \xrightarrow{\text{Evacuate Req 2}} \text{Iteration } t+1: [\text{Req 1 (step 51)}, \text{Req 3 (step 1)}]$$

In [ ]:
class Request:
    def __init__(self, req_id, prompt, max_tokens):
        self.req_id = req_id
        self.prompt = prompt
        self.max_tokens = max_tokens
        self.generated_tokens = 0
        self.is_finished = False

    def step(self):
        self.generated_tokens += 1
        if self.generated_tokens >= self.max_tokens:
            self.is_finished = True
        return f"tok_{self.generated_tokens}"

# Step 2: Implement Continuous Batching Scheduler
class ContinuousBatchScheduler:
    def __init__(self, max_batch_size=2):
        self.max_batch_size = max_batch_size
        self.waiting_queue = []
        self.running_batch = []

    def add_request(self, request):
        self.waiting_queue.append(request)

    def step_iteration(self, iteration_num):
        # 1. Fill empty batch slots from waiting queue
        while len(self.running_batch) < self.max_batch_size and len(self.waiting_queue) > 0:
            admitted = self.waiting_queue.pop(0)
            self.running_batch.append(admitted)
            print(f"  [Iter {iteration_num}] Admitted {admitted.req_id} (target: {admitted.max_tokens} tokens)")

        # 2. Execute 1 token generation step for all running requests
        active_ids = [r.req_id for r in self.running_batch]
        for r in self.running_batch:
            r.step()

        # 3. Evacuate finished requests immediately
        finished = [r for r in self.running_batch if r.is_finished]
        for f in finished:
            print(f"  [Iter {iteration_num}] Finished & Evacuated {f.req_id}!")
            self.running_batch.remove(f)

        return active_ids

# Simulate 3 Requests arriving:
# Req 1: short (2 tokens)
# Req 2: medium (5 tokens)
# Req 3: short (2 tokens, waiting in queue)
scheduler = ContinuousBatchScheduler(max_batch_size=2)
scheduler.add_request(Request("Req_1", "Short query", max_tokens=2))
scheduler.add_request(Request("Req_2", "Medium query", max_tokens=5))
scheduler.add_request(Request("Req_3", "Arrived query", max_tokens=2))

print("--- Continuous Batching Execution Trace ---")
for iter_idx in range(1, 8):
    active = scheduler.step_iteration(iter_idx)
    if not scheduler.running_batch and not scheduler.waiting_queue:
        print(f"Iteration {iter_idx}: All queues drained!")
        break

### 3. Step-by-Step Code Walkthrough
1. **Dynamic Insertion:** At Iteration 2, `Req_1` completed its 2 tokens and was evacuated immediately.
2. **Zero Idle GPU Slots:** At Iteration 3, `Req_3` was admitted into the open slot alongside `Req_2` without waiting for `Req_2` to finish.
3. **Throughput Scaling:** Achieves 2x to 5x higher total token throughput compared to static batching.

---

### 4. Industry Applications & Trade-offs
- **vLLM & TensorRT-LLM:** Core scheduling engine in all high-concurrency production inference endpoints.
- **Trade-off:** Mixing prefill phase (compute-bound) with decode phase (memory-bound) in the same iteration causes interference; mitigated by **Chunked Prefill**.

---

### 5. Top Interview Questions & Answers
**Q1: What is the difference between Prefill phase and Decode phase in LLM inference?**
> **Answer:** **Prefill Phase:** Processes the input prompt all at once, computing attention across all prompt tokens in parallel (Compute-Bound, high GPU utilization). **Decode Phase:** Autoregressively generates one token at a time, generating 1 token per forward pass (Memory-Bound, low GPU utilization).

**Q2: What is Chunked Prefill and why is it necessary in continuous batching?**
> **Answer:** If a user submits a 10,000-token prompt, running prefill in one step monopolizes the GPU, causing severe latency spikes (Time-to-Next-Token) for all ongoing decode streams. Chunked Prefill breaks large prompts into 512-token chunks and interleaves them across decode iterations to maintain steady latency.